# Stablecoins — TitusCoin and the Two Ledgers

Welcome to **TitusCoin (`TTC`)**, a completely fictional token targeting **one fictional USD**. Fictional Titus Bank is fictional too; its lawyers are, presumably, on an equally fictional lunch break.

This notebook is a small learning model, not financial advice, a real reserve, or a production stablecoin. We will follow what happens when an issuer creates, moves, redeems, and then under-backs tokens.

The toy model is ours. The mechanisms are not — see **Sources** at the end.


## Recap

The chain is still a notary. Notebooks 6 and 7 asked it to price ETH; it believed whatever input the lending rule treated as fact. This notebook asks it to back a dollar. Same trust boundary. Different courier: a reserve record the chain cannot see.

| Notebook | What it established |
| --- | --- |
| 5 | Each node has a local mempool. `broadcast`, then `include`. |
| 6–7 | A contract treats some input as fact. The chain notarises the call; it does not audit the universe. |
| 8 | A stablecoin is two ledgers: tokens on-chain, dollars off-chain. The receipt book is not an X-ray of the vault. |

## Build the model in three layers

Notebook 8 defines the stablecoin abstractions in the open. [`blockchain_lib/stablecoin.py`](../blockchain_lib/stablecoin.py) is the copy Notebook 9 imports — there the chain class is named `Blockchain`, same object, shorter label.

The chain underneath is the same public PoS chain as notebooks 5–7; token events travel through the same `Network` (`broadcast`, then `include`).

`FiatBackedIssuer` → `TokenLedger` → `StableCoinBlockchain`

The arrow means **uses**: the issuer asks the ledger to change tokens, and the ledger asks the chain to record the event.

| Layer | Knows | Cannot know |
|---|---|---|
| `StableCoinBlockchain` | blocks, validators, proposer rewards | token balances or whether reserve dollars exist |
| `TokenLedger` | balances, supply, token events | the issuer's bank balance |
| `FiatBackedIssuer` | its reserve record and token ledger | whether a market will value TTC at one dollar |

That separation is the lesson. A pristine receipt book is useful, but it is not an X-ray machine for a bank vault.


In [3]:
import json
import math
import random

from blockchain_lib.mempool import Network, Transaction
from blockchain_lib.pos import Block, Blockchain, Validator

BLOCK_REWARD: float = 2.0
FEE_PER_TX: float = 0.1


### Abstraction 1: `StableCoinBlockchain` — the receipt book

This class **inherits** the canonical proof-of-stake `Blockchain` introduced earlier in Notebook 2. `super().__init__(validators)` lets the parent build the genesis block and validator machinery; this subclass adds a readable `name` and overrides `add_block` with a toy proposer-reward policy. The library copy in `blockchain_lib/stablecoin.py` keeps that subclass and just calls it `Blockchain`, which is what notebook 9 imports.

Its overridden `add_block` and `accept_candidate` pay `BLOCK_REWARD + FEE_PER_TX` to the proposer. Token events arrive through `Network.include`, which calls `accept_candidate`. Despite its name, `FEE_PER_TX` is merely a fixed reward component labelled as a fee: this model has no payer and collects no fee. The inherited chain-validity rules still apply.

The chain owns blocks and validator state. It records token-event receipts, but it neither owns balances nor proves that Fictional Titus Bank has a single fictional dollar.


In [5]:
class StableCoinBlockchain(Blockchain):
    """PoS blockchain extended with a readable name and proposer rewards."""

    def __init__(self, name: str, validators: list[Validator]) -> None:
        """Build the genesis chain via the parent class, then attach a readable name."""
        super().__init__(validators)
        self.name = name

    def add_block(self, data: str) -> tuple[Block, Validator]:
        """Append a block through the shared PoS implementation and pay its proposer."""
        block, proposer = super().add_block(data)
        proposer.stake += BLOCK_REWARD + FEE_PER_TX
        return block, proposer

    def accept_candidate(self, block: Block) -> None:
        """Append a mempool-included candidate and pay its named proposer."""
        super().accept_candidate(block)
        proposer = next(v for v in self.validators if v.name == block.proposer)
        proposer.stake += BLOCK_REWARD + FEE_PER_TX


### Abstraction 2: `TokenLedger` — who owns how many tokens?

The ledger owns three pieces of state: a token `symbol`, a `balances` dictionary, and `total_supply`. It also holds the notebook 5 `Network`. `_record` serializes each event as key-sorted JSON, wraps it in a `Transaction`, then calls `broadcast` and `include` — the same two methods notebook 7 used for the flash-loan attack.

Every public transition first uses `_require_positive`: amounts must be finite and greater than zero, so zero, negatives, `NaN`, and infinities are rejected. Then the accounting rules are:

- `mint`: one balance and total supply rise by the same amount.
- `transfer`: one balance falls as another rises; total supply and the sum of balances are conserved.
- `burn`: one balance and total supply fall by the same amount.

An insufficient balance returns `False` before state or chain history changes. The three HTLC-named methods are merely hooks for Notebook 9: locking removes tokens from the *available* balance, while release or refund restores them later. During escrow, available balances **plus the HTLC manager's escrowed amount** still equal total supply. We are labeling the emergency exits here, not pulling the fire alarm yet.


In [7]:
class TokenLedger:
    """Minimal on-chain token ledger for stablecoin and HTLC lessons."""

    def __init__(self, symbol: str, blockchain: Blockchain, network: Network) -> None:
        """Start an empty ledger for `symbol`, recorded through `network`."""
        self.symbol = symbol
        self.blockchain = blockchain
        self.network = network
        self.balances: dict[str, float] = {}
        self.total_supply: float = 0.0

    @staticmethod
    def _require_positive(amount: float) -> None:
        """Raise if `amount` is not a finite, strictly positive number."""
        if not math.isfinite(amount) or amount <= 0:
            raise ValueError("Amount must be positive.")

    def record_payload(self, payload: str, tx_id: str | None = None) -> None:
        """Gossip `payload` as a Transaction and include it from the origin node."""
        tx = Transaction(tx_id or f"{self.symbol}-{len(self.blockchain.chain)}", payload)
        origin = self.network.node_names[0]
        self.network.broadcast(tx, origin)
        self.network.include(origin, tx.tx_id, self.blockchain)

    def _record(self, record: dict[str, str | float]) -> None:
        """Serialize `record` and include it through the shared mempool."""
        event = str(record.get("event", "EVENT"))
        self.record_payload(
            json.dumps(record, sort_keys=True),
            tx_id=f"{self.symbol}-{event}-{len(self.blockchain.chain)}",
        )

    def mint(self, to: str, amount: float) -> None:
        """Create `amount` new tokens for `to`, raising total supply and recording a MINT."""
        self._require_positive(amount)
        self.balances[to] = self.balances.get(to, 0) + amount
        self.total_supply += amount
        self._record({"event": "MINT", "symbol": self.symbol, "to": to, "amount": amount})

    def transfer(self, sender: str, receiver: str, amount: float) -> bool:
        """Move `amount` from `sender` to `receiver`; return False if sender's balance is short."""
        self._require_positive(amount)
        if self.balances.get(sender, 0) < amount:
            return False
        self.balances[sender] -= amount
        self.balances[receiver] = self.balances.get(receiver, 0) + amount
        self._record(
            {
                "event": "TRANSFER",
                "symbol": self.symbol,
                "from": sender,
                "to": receiver,
                "amount": amount,
            }
        )
        return True

    def burn(self, holder: str, amount: float) -> bool:
        """Destroy `amount` of `holder`'s tokens; return False if their balance is short."""
        self._require_positive(amount)
        if self.balances.get(holder, 0) < amount:
            return False
        self.balances[holder] -= amount
        self.total_supply -= amount
        self._record(
            {"event": "BURN", "symbol": self.symbol, "holder": holder, "amount": amount}
        )
        return True

    def lock_for_htlc(self, holder: str, amount: float, hash_lock: str) -> bool:
        """Move `amount` out of `holder`'s available balance into HTLC escrow (Notebook 9)."""
        self._require_positive(amount)
        if self.balances.get(holder, 0) < amount:
            return False
        self.balances[holder] -= amount
        self._record(
            {
                "event": "HTLC_LOCK",
                "symbol": self.symbol,
                "holder": holder,
                "amount": amount,
                "hash_lock": hash_lock,
            }
        )
        return True

    def release_to(self, receiver: str, amount: float) -> None:
        """Pay escrowed `amount` to `receiver` once an HTLC secret is revealed (Notebook 9)."""
        self._require_positive(amount)
        self.balances[receiver] = self.balances.get(receiver, 0) + amount

    def refund_to(self, sender: str, amount: float) -> None:
        """Return escrowed `amount` to `sender` after an HTLC times out (Notebook 9)."""
        self._require_positive(amount)
        self.balances[sender] = self.balances.get(sender, 0) + amount


### Abstraction 3: `FiatBackedIssuer` — keeper of the off-chain promise

The issuer owns the fictional `reserve_usd` record and collaborates with one `TokenLedger`. It coordinates the two books:

- `deposit_and_mint` validates the amount, raises the reserve record, then asks the ledger to mint the matching tokens.
- `redeem` first checks reserve capacity, then asks the ledger to burn. Only after both preconditions succeed does it reduce the reserve. A failed precheck leaves the books alone.
- `record_reserve_loss` changes only the off-chain reserve, so it deliberately creates no block.

`reserve_ratio` is reserve divided by supply; with zero supply it returns `1.0` by convention rather than dividing by zero. `backing_per_token` caps that ratio at `1.0`, so over-collateralisation does not turn the one-dollar promise into a two-dollar token in this toy model. Neither property is a price oracle—the market remains stubbornly outside this notebook.

In [9]:
class FiatBackedIssuer:
    """Model a fictional issuer's off-chain USD reserve for teaching purposes.

    The reserve is not a real bank balance and is intentionally kept off-chain.
    """

    def __init__(self, name: str, ledger: TokenLedger) -> None:
        """Pair this issuer with the `TokenLedger` it mints into and redeems from."""
        self.name = name
        self.ledger = ledger
        self.reserve_usd: float = 0.0

    def deposit_and_mint(self, to: str, usd_amount: float) -> None:
        """Record fictional USD reserves and mint the matching token amount."""
        TokenLedger._require_positive(usd_amount)
        self.reserve_usd += usd_amount
        self.ledger.mint(to, usd_amount)

    def redeem(self, holder: str, token_amount: float) -> bool:
        """Burn tokens and release the corresponding fictional reserve amount."""
        TokenLedger._require_positive(token_amount)
        if self.reserve_usd < token_amount:
            return False
        if not self.ledger.burn(holder, token_amount):
            return False
        self.reserve_usd -= token_amount
        return True

    def record_reserve_loss(self, usd_amount: float) -> bool:
        """Record an off-chain loss against this fictional issuer's reserves."""
        TokenLedger._require_positive(usd_amount)
        if self.reserve_usd < usd_amount:
            return False
        self.reserve_usd -= usd_amount
        return True

    @property
    def reserve_ratio(self) -> float:
        """Return the fictional reserve amount divided by issued token supply."""
        if self.ledger.total_supply == 0:
            return 1.0
        return self.reserve_usd / self.ledger.total_supply

    @property
    def backing_per_token(self) -> float:
        """Return simplified fictional backing per token, not a market-price oracle."""
        return min(1.0, self.reserve_ratio)

### Wire the collaborators together

Now we create the receipt book, the same `Network` as notebooks 5–7, hand both to the token ledger, and hand that ledger to the issuer. A call such as `deposit_and_mint` travels issuer → ledger → `broadcast` / `include` → chain.


In [11]:
nodes = ["Titus-Node-1", "Titus-Node-2"]
network = Network(nodes, random.Random(7))
chain = StableCoinBlockchain(
    "TitusChain",
    [Validator("Titus-Node-1", 250), Validator("Titus-Node-2", 150)],
)
tituscoin = TokenLedger("TTC", chain, network)
titus_issuer = FiatBackedIssuer("Fictional Titus Bank", tituscoin)


## 1. Two ledgers, one promise

A fiat-backed token has two different kinds of state:

- `TokenLedger` owns the **on-chain token ledger**: `TTC` balances and total supply. Its `_record` helper gossips a `Transaction` and includes it on `StableCoinBlockchain`.
- `FiatBackedIssuer` owns the **off-chain reserve record**: how many fictional dollars Fictional Titus Bank says it holds.

The chain cannot peek into a bank vault. Keeping these books separate is the important part: an immaculate chain does not prove the reserve exists. Both begin at zero.


In [13]:
print(f"Token supply: {tituscoin.total_supply:.0f} TTC")
print(f"Off-chain reserve: ${titus_issuer.reserve_usd:,.0f} fictional USD")

Token supply: 0 TTC
Off-chain reserve: $0 fictional USD


## 2. Issuance: deposit first, mint second

Alice deposits 1,000 fictional USD with the issuer. `FiatBackedIssuer.deposit_and_mint` raises its reserve record, then calls `TokenLedger.mint` to create 1,000 TTC for Alice. `_record` puts the `MINT` receipt on TitusChain; the reserve entry remains an off-chain claim. No suitcase of dollars is squeezed into a block.

In [15]:
titus_issuer.deposit_and_mint("Alice", 1_000)

print(f"Alice: {tituscoin.balances['Alice']:.0f} TTC")
print(f"Supply: {tituscoin.total_supply:.0f} TTC")
print(f"Reserve: ${titus_issuer.reserve_usd:,.0f} fictional USD")

Alice: 1000 TTC
Supply: 1000 TTC
Reserve: $1,000 fictional USD


## 3. Transfer: ownership moves, supply does not

> **Pause and predict:** Alice sends Bob 250 TTC. What should change: their balances, total supply, the fictional USD reserve, or some combination?

`TokenLedger.transfer` only reassigns existing tokens: it subtracts and adds the same amount, preserving both total supply and the sum of balances. It never calls the issuer, so `reserve_usd` stays put because nobody redeemed anything.

In [17]:
supply_before_transfer = tituscoin.total_supply
reserve_before_transfer = titus_issuer.reserve_usd
transfer_succeeded = tituscoin.transfer("Alice", "Bob", 250)

assert transfer_succeeded
assert tituscoin.total_supply == supply_before_transfer
assert titus_issuer.reserve_usd == reserve_before_transfer
assert sum(tituscoin.balances.values()) == tituscoin.total_supply
print(tituscoin.balances)
print(f"Supply still: {tituscoin.total_supply:.0f} TTC")

{'Alice': 750, 'Bob': 250}
Supply still: 1000 TTC


## 4. Redemption: burn a token, release a dollar

> **Pause and predict:** Bob redeems 100 TTC. After the operation, what are Bob's balance, total supply, and the issuer's reserve?

`FiatBackedIssuer.redeem` coordinates both books. It checks the reserve, calls `TokenLedger.burn` to remove Bob's 100 TTC on-chain, and only then reduces `reserve_usd` by 100 off-chain. In this model those values move together, so fully backed tokens remain fully backed.

In [19]:
redemption_succeeded = titus_issuer.redeem("Bob", 100)

assert redemption_succeeded
print(f"Bob: {tituscoin.balances['Bob']:.0f} TTC")
print(f"Supply: {tituscoin.total_supply:.0f} TTC")
print(f"Reserve: ${titus_issuer.reserve_usd:,.0f} fictional USD")

Bob: 150 TTC
Supply: 900 TTC
Reserve: $900 fictional USD


## 5. Reserve loss: an off-chain problem does not rewrite the token ledger

> **Pause and predict:** Fictional Titus Bank records a 180 fictional USD reserve loss. Do token balances or total supply automatically shrink? What happens to backing per token?

`FiatBackedIssuer.record_reserve_loss` changes only `reserve_usd`. It never calls the ledger, so it neither burns Alice's or Bob's tokens nor asks the chain to add a block. The chain can remain cryptographically valid while the economic promise weakens. Awkward, but educationally useful.

In [21]:
supply_before_loss = tituscoin.total_supply
balances_before_loss = tituscoin.balances.copy()
blocks_before_loss = len(chain.chain)
loss_recorded = titus_issuer.record_reserve_loss(180)

assert loss_recorded
assert tituscoin.total_supply == supply_before_loss
assert tituscoin.balances == balances_before_loss
assert len(chain.chain) == blocks_before_loss
print(f"Reserve after loss: ${titus_issuer.reserve_usd:,.0f} fictional USD")
print(f"Supply after loss: {tituscoin.total_supply:.0f} TTC")

Reserve after loss: $720 fictional USD
Supply after loss: 900 TTC


## 6. Reserve ratio is not market price

`FiatBackedIssuer.reserve_ratio` computes `reserve / token supply`: here, 720 / 900 = 0.80. Its `backing_per_token` property therefore reports 0.80 fictional USD of simplified backing per TTC.

That is an accounting ratio, **not a market price**. A real trading price comes from buyers, sellers, liquidity, redemption confidence, information, and sometimes collective eyebrow-raising. This notebook has no market, order book, oracle, or price-discovery mechanism, so it cannot calculate a depeg price.

In [23]:
print(f"Reserve ratio: {titus_issuer.reserve_ratio:.0%}")
print(
    f"Simplified backing: ${titus_issuer.backing_per_token:.2f} "
    "fictional USD per TTC"
)
print(f"Chain check: {chain.is_valid()}")

Reserve ratio: 80%
Simplified backing: $0.80 fictional USD per TTC
Chain check: (True, 'Chain is valid.')


## 7. What this pocket-sized model leaves out

Real fiat-backed tokens need much more than a tidy Python dictionary: custody, independent attestations or audits, legal redemption rights, banking access, access control, identity and compliance processes, operational security, fees, token decimals, failure handling, and trustworthy reporting from off-chain systems.

Our model also assumes every accepted reserve entry is true and every successful redemption pays exactly one fictional USD per token. It demonstrates **state relationships**, not proof of solvency, safety, liquidity, or price stability.

In [25]:
print("Final fictional snapshot")
print(f"  balances: {tituscoin.balances}")
print(f"  supply: {tituscoin.total_supply:.0f} TTC")
print(f"  reserve: ${titus_issuer.reserve_usd:,.0f}")
print(f"  backing: ${titus_issuer.backing_per_token:.2f} per TTC")

Final fictional snapshot
  balances: {'Alice': 750, 'Bob': 150}
  supply: 900 TTC
  reserve: $720
  backing: $0.80 per TTC


## Next: two chains, one nervous exchange

We can now issue and account for a token on one chain. **Notebook 9 asks the next question: how can Alice and Bob exchange tokens across two independent chains so that both sides settle—or both sides can safely unwind—without trusting one party to go first?**

## Sources

A token that claims to be a dollar, with a reserve the chain cannot see, is how fiat-backed stablecoins actually work:

- Gorton, G. B., & Zhang, J. Y. (2023). [Taming Wildcat Stablecoins](https://lawreview.uchicago.edu/print-archive/taming-wildcat-stablecoins). *University of Chicago Law Review*. Redeemable private money, run risk, and why the issuer's books are not on-chain.
- Adrian, T., & Mancini-Griffoli, T. (2019). [The Rise of Digital Money](https://www.imf.org/en/Publications/fintech-notes/Issues/2019/07/12/The-Rise-of-Digital-Money-47097). IMF Fintech Note. How e-money / stablecoin claims sit next to bank deposits.
- Bank for International Settlements / CPMI-IOSCO (2022). [Application of the Principles for Financial Market Infrastructures to stablecoin arrangements](https://www.bis.org/cpmi/publ/d206.htm). What "backed 1:1" still has to prove when the reserve is off-chain.
